# NYC Taxi Trip Duration Prediction

**CMPE 255 — Assignment 1, Part 2**

This notebook builds a reproducible supervised-regression experiment inspired by Kaggle's NYC Taxi Trip Duration challenge. It follows CRISP-DM from business framing through evaluation and conclusion. **All reported results are computed from the real `train.csv`; none are pre-filled or fabricated.**

## 1. Business Understanding

The objective is to predict `trip_duration` in seconds using facts known when a trip begins. Accurate estimates may improve rider ETAs, dispatch decisions, and capacity planning.

**Success criteria**
- Beat a median-prediction baseline on held-out MAE.
- Use **MAE** as the primary, readily interpretable metric; inspect **RMSE** for sensitivity to large errors and **R²** for explained variance.
- Avoid leakage: `dropoff_datetime` is not available at pickup and must never be a predictor.
- Prefer the simplest model when performance is effectively comparable, while also considering latency and interpretability.

This is an offline experiment, not a production ETA service.

## 2. Data Understanding

The expected source is Kaggle's [NYC Taxi Trip Duration competition](https://www.kaggle.com/competitions/nyc-taxi-trip-duration/data). Download `train.csv` and place it **beside this notebook**. If the file is absent, the next cell prints actionable instructions and data-dependent cells safely skip; synthetic data is never substituted.

The original training schema includes identifiers, vendor/passenger data, pickup/dropoff timestamps and coordinates, a forwarding flag, and the target `trip_duration`. To keep compute reasonable, this experiment uses all rows when there are at most 75,000, otherwise a reproducible 75,000-row sample.

In [ ]:
from pathlib import Path
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
RANDOM_STATE = 42
SAMPLE_SIZE = 75_000
TEST_SIZE = 0.20
DATA_PATH = Path("train.csv")
IMAGE_DIR = Path("images")
IMAGE_DIR.mkdir(exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
DATA_AVAILABLE = DATA_PATH.exists()

if not DATA_AVAILABLE:
    print(
        "DATASET NOT FOUND — template mode enabled.\n"
        "Download train.csv from the Kaggle NYC Taxi Trip Duration competition:\n"
        "https://www.kaggle.com/competitions/nyc-taxi-trip-duration/data\n"
        "Then place it at:\n"
        f"  {DATA_PATH.resolve()}\n"
        "Restart the kernel and run all cells. Data-dependent cells will be skipped "
        "until the file is supplied; no synthetic data or results will be created."
    )
else:
    required_columns = {
        "pickup_datetime", "pickup_longitude", "pickup_latitude",
        "dropoff_longitude", "dropoff_latitude", "passenger_count", "trip_duration"
    }
    raw = pd.read_csv(DATA_PATH)
    missing_columns = required_columns.difference(raw.columns)
    if missing_columns:
        raise ValueError(
            "The supplied train.csv does not match the expected Kaggle training schema. "
            f"Missing required columns: {sorted(missing_columns)}"
        )

    if len(raw) > SAMPLE_SIZE:
        df = raw.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).copy()
        sample_note = f"Reproducibly sampled {SAMPLE_SIZE:,} of {len(raw):,} rows."
    else:
        df = raw.copy()
        sample_note = f"Using all {len(raw):,} rows (below the {SAMPLE_SIZE:,}-row cap)."
    del raw
    print(sample_note)
    print(f"Shape: {df.shape}")
    display(df.head())


In [ ]:
if not DATA_AVAILABLE:
    print("Skipped data type summary: supply train.csv and rerun the notebook.")
else:
    df.info()


In [ ]:
if not DATA_AVAILABLE:
    print("Skipped descriptive statistics: supply train.csv and rerun the notebook.")
else:
    df.describe(include="all").T


## 3. Exploratory Data Analysis

EDA is descriptive only. It is performed before modeling, while any operation that *learns parameters* (imputation, scaling, model fitting) is deferred until after the split. Plots are saved to `images/` as well as displayed.

In [ ]:
if not DATA_AVAILABLE:
    print("Skipped exploratory visualizations: supply train.csv and rerun the notebook.")
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(df["trip_duration"].dropna(), bins=80, color="#2878B5", alpha=0.85)
    ax.set(title="Raw trip-duration distribution", xlabel="Trip duration (seconds)", ylabel="Trips")
    fig.tight_layout()
    fig.savefig(IMAGE_DIR / "target_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

    pickup_preview = pd.to_datetime(df["pickup_datetime"], errors="coerce")
    hour_counts = pickup_preview.dt.hour.value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(hour_counts.index, hour_counts.values, color="#45A778")
    ax.set(title="Trips by pickup hour", xlabel="Hour", ylabel="Trips", xticks=range(0, 24, 2))
    fig.tight_layout()
    fig.savefig(IMAGE_DIR / "trips_by_hour.png", dpi=150, bbox_inches="tight")
    plt.show()


## 4. Data Cleaning

Cleaning is intentionally transparent rather than silently clipping values. Datetimes and numeric fields are coerced, then records that cannot support the analysis are removed. Plausibility rules focus on the competition's NYC context:

- positive duration, capped at 24 hours;
- passenger count from 1 through 8;
- valid global coordinate ranges, then an NYC-area bounding box;
- engineered straight-line distance no greater than 100 km.

These are modeling assumptions, not universal truths; sensitivity checks are recommended before deployment.

### 4.1 Missing-value analysis

In [ ]:
if not DATA_AVAILABLE:
    print("Skipped missing-value analysis: supply train.csv and rerun the notebook.")
else:
    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean().mul(100)
    }).sort_values("missing_count", ascending=False)
    missing.loc[missing["missing_count"] > 0] if missing["missing_count"].any() else pd.DataFrame(
        {"status": ["No explicit missing values before type coercion"]}
    )


### 4.2 Duplicate analysis

Both exact-row duplicates and duplicate trip IDs (when `id` exists) are inspected. Exact duplicates are removed. Repeated IDs with differing content are also removed after keeping the first occurrence, because an ID should identify one trip.

In [ ]:
if not DATA_AVAILABLE:
    print("Skipped duplicate analysis: supply train.csv and rerun the notebook.")
else:
    exact_duplicates = int(df.duplicated().sum())
    id_duplicates = int(df["id"].duplicated().sum()) if "id" in df.columns else None
    print(f"Exact duplicate rows: {exact_duplicates:,}")
    print("Duplicate IDs:", f"{id_duplicates:,}" if id_duplicates is not None else "id column unavailable")

    df = df.drop_duplicates().copy()
    if "id" in df.columns:
        df = df.drop_duplicates(subset="id", keep="first").copy()


### 4.3 Outlier analysis

Quantiles are shown before applying domain filters. A post-cleaning IQR count is also reported for transparency, but IQR points are **not automatically deleted**: traffic can create legitimate long trips.

In [ ]:
if not DATA_AVAILABLE:
    print("Skipped outlier quantiles: supply train.csv and rerun the notebook.")
else:
    numeric_review = [
        "trip_duration", "passenger_count", "pickup_longitude", "pickup_latitude",
        "dropoff_longitude", "dropoff_latitude"
    ]
    df[numeric_review].quantile([0, .01, .25, .5, .75, .99, 1]).T


## 5. Feature Engineering

Pickup time yields hour, day of week (Monday=0), and month. The Haversine formula converts coordinate pairs to great-circle distance in kilometers. This captures geography without using post-pickup information.

In [ ]:
if not DATA_AVAILABLE:
    print("Skipped cleaning and feature engineering: supply train.csv and rerun the notebook.")
else:
    def haversine_km(lon1, lat1, lon2, lat2):
        """Vectorized great-circle distance in kilometers."""
        lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
        dlon, dlat = lon2 - lon1, lat2 - lat1
        a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
        return 2 * 6371.0088 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

    numeric_columns = [
        "pickup_longitude", "pickup_latitude", "dropoff_longitude",
        "dropoff_latitude", "passenger_count", "trip_duration"
    ]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce")

    df["pickup_hour"] = df["pickup_datetime"].dt.hour
    df["pickup_dayofweek"] = df["pickup_datetime"].dt.dayofweek
    df["pickup_month"] = df["pickup_datetime"].dt.month
    df["distance_km"] = haversine_km(
        df["pickup_longitude"], df["pickup_latitude"],
        df["dropoff_longitude"], df["dropoff_latitude"]
    )

    complete_required = df[["pickup_datetime", "trip_duration"]].notna().all(axis=1)
    valid = (
        complete_required
        & df["trip_duration"].between(1, 86_400)
        & df["passenger_count"].between(1, 8)
        & df["pickup_longitude"].between(-75, -72)
        & df["dropoff_longitude"].between(-75, -72)
        & df["pickup_latitude"].between(40, 42)
        & df["dropoff_latitude"].between(40, 42)
        & df["distance_km"].between(0, 100)
    )
    clean_df = df.loc[valid].copy()
    print(f"Rows before domain cleaning: {len(df):,}")
    print(f"Rows removed: {(~valid).sum():,} ({(~valid).mean():.2%})")
    print(f"Rows retained: {len(clean_df):,}")
    if len(clean_df) < 100:
        raise ValueError("Fewer than 100 valid rows remain; inspect the source data and cleaning assumptions.")

    q1, q3 = clean_df["trip_duration"].quantile([.25, .75])
    iqr = q3 - q1
    iqr_outliers = ((clean_df["trip_duration"] < q1 - 1.5 * iqr) |
                    (clean_df["trip_duration"] > q3 + 1.5 * iqr)).sum()
    print(f"Post-cleaning target observations outside 1.5×IQR: {iqr_outliers:,} (retained)")


In [ ]:
if not DATA_AVAILABLE:
    print("Skipped distance-versus-duration plot: supply train.csv and rerun the notebook.")
else:
    plot_sample = clean_df.sample(n=min(10_000, len(clean_df)), random_state=RANDOM_STATE)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(plot_sample["distance_km"], plot_sample["trip_duration"] / 60,
               s=8, alpha=.2, color="#D65F5F")
    ax.set(title="Distance versus trip duration", xlabel="Haversine distance (km)", ylabel="Duration (minutes)")
    fig.tight_layout()
    fig.savefig(IMAGE_DIR / "distance_vs_duration.png", dpi=150, bbox_inches="tight")
    plt.show()


## 6. Train/Test Split and Leakage Prevention

Only features available at pickup are selected. The split occurs before learned imputation, scaling, or fitting. Those transformations live inside pipelines and are fit exclusively on training rows. A fixed 80/20 split makes comparisons reproducible.

The target is modeled as `log1p(trip_duration)` via `TransformedTargetRegressor`; predictions are automatically returned in seconds. This reduces target skew without changing the metrics' units.

In [ ]:
if not DATA_AVAILABLE:
    print("Skipped train/test split: supply train.csv and rerun the notebook.")
else:
    feature_columns = [
        "pickup_hour", "pickup_dayofweek", "pickup_month", "passenger_count",
        "pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude",
        "distance_km"
    ]
    X = clean_df[feature_columns].copy()
    y = clean_df["trip_duration"].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
    print("Leakage check — excluded columns:", sorted(set(clean_df.columns) - set(feature_columns)))


## 7. Modeling

Four regressors are compared under the same split. The dummy model establishes the minimum useful benchmark. Tree complexity and iterations are bounded to keep the experiment practical.

In [ ]:
if not DATA_AVAILABLE:
    print("Skipped model definitions: supply train.csv and rerun the notebook.")
else:
    def pipeline(model, scale=False):
        steps = [("imputer", SimpleImputer(strategy="median"))]
        if scale:
            steps.append(("scaler", StandardScaler()))
        steps.append(("model", model))
        return Pipeline(steps)

    models = {
        "Median baseline": pipeline(DummyRegressor(strategy="median")),
        "Linear Regression": TransformedTargetRegressor(
            regressor=pipeline(LinearRegression(), scale=True),
            func=np.log1p, inverse_func=np.expm1
        ),
        "Random Forest": TransformedTargetRegressor(
            regressor=pipeline(RandomForestRegressor(
                n_estimators=120, max_depth=18, min_samples_leaf=3,
                n_jobs=-1, random_state=RANDOM_STATE
            )), func=np.log1p, inverse_func=np.expm1
        ),
        "HistGradientBoosting": TransformedTargetRegressor(
            regressor=pipeline(HistGradientBoostingRegressor(
                max_iter=180, learning_rate=.08, max_leaf_nodes=31,
                l2_regularization=1.0, random_state=RANDOM_STATE
            )), func=np.log1p, inverse_func=np.expm1
        )
    }


## 8. Model Evaluation and Comparison

Metrics are computed only on the untouched test partition. Lower MAE/RMSE is better; higher R² is better.

In [ ]:
if not DATA_AVAILABLE:
    print("Skipped model training and evaluation: supply train.csv and rerun the notebook.")
else:
    results = []
    predictions = {}
    for name, model in models.items():
        started = time.perf_counter()
        model.fit(X_train, y_train)
        pred = np.maximum(model.predict(X_test), 0)
        elapsed = time.perf_counter() - started
        predictions[name] = pred
        results.append({
            "Model": name,
            "MAE (seconds)": mean_absolute_error(y_test, pred),
            "RMSE (seconds)": mean_squared_error(y_test, pred) ** 0.5,
            "R²": r2_score(y_test, pred),
            "Fit + predict (s)": elapsed,
        })

    results_df = pd.DataFrame(results).sort_values("MAE (seconds)").reset_index(drop=True)
    results_df.round(3)


In [ ]:
if not DATA_AVAILABLE:
    print("Skipped model comparison plot: supply train.csv and rerun the notebook.")
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ordered = results_df.sort_values("MAE (seconds)")
    ax.barh(ordered["Model"], ordered["MAE (seconds)"], color="#6F4E7C")
    ax.invert_yaxis()
    ax.set(title="Held-out model comparison", xlabel="MAE (seconds)", ylabel="")
    fig.tight_layout()
    fig.savefig(IMAGE_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()


### 8.1 Actual vs. predicted visualization

The best model is selected by held-out MAE. The scatter is sampled only for rendering speed; the metrics above use the complete test set.

In [ ]:
if not DATA_AVAILABLE:
    print("Skipped actual-versus-predicted diagnostics: supply train.csv and rerun the notebook.")
else:
    best_name = results_df.loc[0, "Model"]
    best_pred = predictions[best_name]
    rng = np.random.default_rng(RANDOM_STATE)
    plot_idx = rng.choice(len(y_test), size=min(5_000, len(y_test)), replace=False)
    actual_plot = y_test.to_numpy()[plot_idx]
    pred_plot = best_pred[plot_idx]
    plot_limit = np.quantile(np.concatenate([actual_plot, pred_plot]), .99)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(actual_plot, pred_plot, s=9, alpha=.25, color="#2878B5")
    axes[0].plot([0, plot_limit], [0, plot_limit], "--", color="black", linewidth=1)
    axes[0].set(xlim=(0, plot_limit), ylim=(0, plot_limit), title=f"Actual vs predicted — {best_name}",
                xlabel="Actual duration (seconds)", ylabel="Predicted duration (seconds)")
    residuals = actual_plot - pred_plot
    axes[1].scatter(pred_plot, residuals, s=9, alpha=.25, color="#D65F5F")
    axes[1].axhline(0, linestyle="--", color="black", linewidth=1)
    axes[1].set(title="Residual diagnostic", xlabel="Predicted duration (seconds)", ylabel="Actual − predicted (seconds)")
    fig.tight_layout()
    fig.savefig(IMAGE_DIR / "actual_vs_predicted.png", dpi=150, bbox_inches="tight")
    plt.show()


### 8.2 Feature importance where supported

Random forests expose impurity-based feature importance. This is useful for model inspection but does not establish causality and may favor continuous/high-cardinality variables.

In [ ]:
if not DATA_AVAILABLE:
    print("Skipped feature importance: supply train.csv and rerun the notebook.")
else:
    rf_wrapper = models["Random Forest"]
    rf_pipeline = rf_wrapper.regressor_
    rf_model = rf_pipeline.named_steps["model"]
    importance = pd.Series(rf_model.feature_importances_, index=feature_columns).sort_values()

    fig, ax = plt.subplots(figsize=(8, 5))
    importance.plot.barh(ax=ax, color="#45A778")
    ax.set(title="Random Forest feature importance", xlabel="Impurity-based importance", ylabel="")
    fig.tight_layout()
    fig.savefig(IMAGE_DIR / "feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()
    importance.sort_values(ascending=False).to_frame("importance")


## 9. Final Model Recommendation

The following cell makes a recommendation **only from measured held-out results**. Operational selection should additionally consider inference latency, monitoring, interpretability, and whether the observed improvement is meaningful for the business.

In [ ]:
if not DATA_AVAILABLE:
    print("Skipped final model recommendation: supply train.csv and rerun the notebook.")
else:
    best_row = results_df.iloc[0]
    baseline_mae = results_df.loc[results_df["Model"] == "Median baseline", "MAE (seconds)"].iloc[0]
    improvement = baseline_mae - best_row["MAE (seconds)"]
    print(
        f"Recommend {best_row['Model']} based on the lowest held-out MAE "
        f"({best_row['MAE (seconds)']:.2f} seconds), an improvement of "
        f"{improvement:.2f} seconds over the median baseline. "
        "Validate this choice on newer, geographically representative data before deployment."
    )


## 10. Limitations

- Haversine distance is straight-line distance, not routed road distance.
- The competition data represents a particular place and period; temporal and geographic drift can reduce generalization.
- Weather, holidays, traffic, road closures, route choice, and detailed road-network context are absent.
- A random split can place nearby times in both partitions and may be more optimistic than a time-based deployment simulation.
- Fixed outlier and NYC-boundary rules can exclude unusual but genuine trips.
- Impurity-based feature importance is associative, model-specific, and not causal.
- A sample lowers compute but may omit rare regimes; the held-out set estimates performance only for this sampled distribution.

## 11. Future Improvements

1. Use a chronological validation split and cross-validation across time windows.
2. Add rush-hour, weekend, holiday, weather, airport, and geospatial cluster features using pickup-time-safe data.
3. Replace Haversine distance with road-network route distance and estimated route characteristics.
4. Tune hyperparameters on training folds, preserving the test set for one final evaluation.
5. Evaluate segment-level errors (hour, borough/region, distance band) and prediction intervals.
6. Compare modern gradient boosting libraries if assignment constraints permit.
7. Track data drift, latency, and error after deployment and define retraining triggers.

## 12. CRISP-DM Conclusion

This notebook connects the business need for pickup-time duration estimates to data inspection, explicit cleaning, leakage-aware preparation, reproducible modeling, and held-out evaluation. The baseline establishes whether learned models add value; MAE, RMSE, R², diagnostic plots, and feature importance provide complementary evidence. The recommendation above is generated only after execution on the real Kaggle data. Before deployment, the chosen model should be validated chronologically, tested across operational segments, and monitored for drift—continuing the iterative CRISP-DM cycle rather than treating this notebook as its endpoint.